In [ ]:
# bias_assessment.py
# Run this in Colab/local Python. If in Colab, optionally run:
#   !pip install xgboost shap
# before running to enable SHAP TreeExplainer for XGBoost.

import os, warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from scipy.stats import ttest_ind
import matplotlib.pyplot as plt
import importlib.util

# ---------- CONFIG ----------
DATA_PATH = "/mnt/data/92da2bec-b99c-4f8e-91ad-b02d9438d5be.csv"
OUT_DIR = "./bias_assessment_outputs"
os.makedirs(OUT_DIR, exist_ok=True)

# ---------- LOAD ----------
df = pd.read_csv(DATA_PATH)
target = "defaulted" if "defaulted" in df.columns else df.columns[-1]
X = df.drop(columns=[target])
y = df[target].astype(int)

features = list(X.columns)

# ---------- PREPROCESS ----------
imp = SimpleImputer(strategy="median")
X_imp = pd.DataFrame(imp.fit_transform(X), columns=features)

# Standardize for the model (not strictly required for tree models but keeps things consistent)
scaler = StandardScaler().fit(X_imp)
X_scaled = scaler.transform(X_imp)

# ---------- MODEL ----------
# Use XGBoost if installed, otherwise a fast GradientBoosting tree
if importlib.util.find_spec("xgboost") is not None:
    import xgboost as xgb
    model = xgb.XGBClassifier(use_label_encoder=False, eval_metric="logloss",
                              random_state=42, n_estimators=200, max_depth=4, n_jobs=-1)
    model_name = "XGBoost"
else:
    model = GradientBoostingClassifier(n_estimators=150, max_depth=4, random_state=42)
    model_name = "GradientBoosting"

print("Training model:", model_name)
model.fit(X_scaled, y)

# ---------- BASE PROBABILITIES ----------
if hasattr(model, "predict_proba"):
    base_probs = model.predict_proba(X_scaled)[:,1]
else:
    # fallback to decision_function scaled
    scores = model.decision_function(X_scaled)
    base_probs = (scores - scores.min()) / (scores.max() - scores.min() + 1e-9)

# ---------- LOCAL APPROXIMATE CONTRIBUTIONS ----------
# For each feature: contribution = base_prob - prob_with_feature_replaced_by_train_mean
train_means = X_imp.mean()
n = X_imp.shape[0]
m = len(features)
contribs = np.zeros((n, m))

print("Computing per-instance approximate contributions (this will take ~O(n_features * n_instances) model predictions).")
for i, feat in enumerate(features):
    X_mod = X_imp.copy()
    X_mod[feat] = train_means[feat]
    X_mod_s = scaler.transform(X_mod)
    if hasattr(model, "predict_proba"):
        mod_probs = model.predict_proba(X_mod_s)[:,1]
    else:
        scores = model.decision_function(X_mod_s)
        mod_probs = (scores - scores.min()) / (scores.max() - scores.min() + 1e-9)
    contribs[:, i] = base_probs - mod_probs  # positive => feature increases default risk

contribs_df = pd.DataFrame(contribs, columns=features)
contribs_df['defaulted'] = y.values
contribs_df['base_prob'] = base_probs

# Save per-instance contributions for audit
per_instance_csv = os.path.join(OUT_DIR, "per_instance_contributions.csv")
contribs_df.to_csv(per_instance_csv, index=False)
print("Saved per-instance contributions to:", per_instance_csv)

# ---------- GROUP COMPARISON ----------
means_defaulted = contribs_df[contribs_df['defaulted']==1][features].mean()
means_nondefaulted = contribs_df[contribs_df['defaulted']==0][features].mean()
diff = means_defaulted - means_nondefaulted

summary = pd.DataFrame({
    "feature": features,
    "mean_contrib_defaulted": means_defaulted.values,
    "mean_contrib_nondefaulted": means_nondefaulted.values,
    "difference (defaulted - nondefaulted)": diff.values
})

# t-tests (Welch)
pvals = []
for feat in features:
    a = contribs_df[contribs_df['defaulted']==1][feat].values
    b = contribs_df[contribs_df['defaulted']==0][feat].values
    try:
        stat, p = ttest_ind(a, b, equal_var=False, nan_policy='omit')
    except Exception:
        p = np.nan
    pvals.append(p)
summary['p_value_ttest'] = pvals

summary = summary.sort_values("difference (defaulted - nondefaulted)", ascending=False).reset_index(drop=True)

summary_csv = os.path.join(OUT_DIR, "bias_assessment_summary.csv")
summary.to_csv(summary_csv, index=False)
print("Saved summary to:", summary_csv)

# ---------- PLOT ----------
fig, ax = plt.subplots(figsize=(10,6))
idx = np.arange(len(features))
width = 0.4
ax.barh(idx - width/2, means_nondefaulted.values, height=width, label='Non-defaulted (approved) mean contrib')
ax.barh(idx + width/2, means_defaulted.values, height=width, label='Defaulted mean contrib')
ax.set_yticks(idx); ax.set_yticklabels(features)
ax.invert_yaxis()
ax.set_xlabel("Average approx contribution (positive => increases default risk)")
ax.set_title("Bias assessment — average feature contribution by group")
ax.legend()
plt.tight_layout()
plot_path = os.path.join(OUT_DIR, "bias_assessment_group_means.png")
plt.savefig(plot_path)
plt.close()
print("Saved plot to:", plot_path)

# ---------- TOP DIFFERENCES (human-friendly) ----------
top_diff = summary.copy().loc[summary['difference (defaulted - nondefaulted)'].abs().sort_values(ascending=False).index].head(10)
print("\nTop differences (feature, diff, p-value):")
for _, r in top_diff.iterrows():
    print(f" - {r['feature']}: diff={r['difference (defaulted - nondefaulted)']:.4f}, p={r['p_value_ttest']:.3g}, mean_defaulted={r['mean_contrib_defaulted']:.4f}, mean_nondefaulted={r['mean_contrib_nondefaulted']:.4f}")

print("\nDone. Review the CSV and PNG in the output directory for more detail:", OUT_DIR)
